# Schedule Revision and Reconciliation for Belgium aFRR

This tutorial walks through the strongest stable `v0.7.0` path in the repo: a Belgium portfolio run that starts with a D-1 `da_plus_afrr` plan, revises future **energy** at public checkpoints while keeping awarded aFRR commitments locked, and then reconciles realized outcomes against baseline and revised expectations.


## What this notebook covers

1. Load the canonical Belgium full-stack config
2. Run a checkpoint-based `schedule_revision` benchmark on top of `da_plus_afrr`
3. Inspect how baseline and revised schedules differ while aFRR commitments stay fixed
4. Reconcile expected vs realized PnL and export downstream artifacts


In [ ]:
from __future__ import annotations

import json
import tempfile
from pathlib import Path

import pandas as pd

from euroflex_bess_lab import (
    export_bids,
    export_revision,
    export_schedule,
    load_config,
    reconcile_run,
    run_walk_forward,
)
from euroflex_bess_lab.analytics.reporting import load_report_summary

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "examples").exists():
    REPO_ROOT = REPO_ROOT.parent

CONFIG_PATH = REPO_ROOT / "examples" / "configs" / "canonical" / "belgium_full_stack.yaml"
ARTIFACT_ROOT = Path(tempfile.mkdtemp(prefix="euroflex-v07-notebook-"))
pd.options.display.max_columns = 30

CONFIG_PATH, ARTIFACT_ROOT

## Step 1 - Load the canonical config

The canonical config is a Belgium portfolio `schedule_revision` run. It wraps `base_workflow: da_plus_afrr`, so the baseline D-1 solve awards asymmetric reserve, while revision checkpoints only reshape future unlocked energy around those fixed reserve commitments.


In [ ]:
config = load_config(CONFIG_PATH)
config.artifacts.root_dir = ARTIFACT_ROOT / "runs"

pd.DataFrame(
    {
        "field": [
            "workflow",
            "base_workflow",
            "market",
            "run_scope",
            "asset_count",
            "site_id",
        ],
        "value": [
            config.workflow,
            config.execution_workflow,
            config.market.id,
            config.run_scope,
            len(config.assets),
            config.site.id,
        ],
    }
)

## Step 2 - Run the walk-forward engine

This produces the baseline plan, revision schedule, schedule lineage, dispatch artifacts, and stable summary. Because the example uses frozen public data, the run is deterministic and safe for notebook execution.


In [ ]:
result = run_walk_forward(config)
summary = load_report_summary(result.output_dir)

summary_frame = pd.DataFrame(
    [
        {
            "run_id": summary["run_id"],
            "workflow": summary["workflow"],
            "base_workflow": summary.get("base_workflow", summary["workflow"]),
            "benchmark_name": summary["benchmark_name"],
            "run_scope": summary["run_scope"],
            "expected_pnl_eur": summary["expected_total_pnl_eur"],
            "realized_pnl_eur": summary.get("realized_total_pnl_eur", summary["total_pnl_eur"]),
            "afrr_capacity_revenue_eur": summary.get("reserve_capacity_revenue_eur", 0.0),
            "afrr_activation_revenue_eur": summary.get("reserve_activation_revenue_eur", 0.0),
            "reserved_up_avg_mw": summary.get("afrr_up_reserved_mw_avg", 0.0),
            "reserved_down_avg_mw": summary.get("afrr_down_reserved_mw_avg", 0.0),
        }
    ]
)
summary_frame

## Step 3 - Inspect baseline vs revised schedules

The most important behavior in `v0.7.0` is that aFRR up/down commitments remain locked after the D-1 award, while future unlocked **energy** can still move at revision checkpoints. The table below surfaces exactly that pattern.


In [ ]:
baseline = pd.read_parquet(result.output_dir / "baseline_schedule.parquet")
revision = pd.read_parquet(result.output_dir / "revision_schedule.parquet")
lineage = pd.read_parquet(result.output_dir / "schedule_lineage.parquet")
decision_log = pd.read_parquet(result.output_dir / "decision_log.parquet")

comparison = baseline[
    [
        "timestamp_local",
        "net_export_mw",
        "afrr_up_reserved_mw",
        "afrr_down_reserved_mw",
        "lock_state",
        "schedule_version",
    ]
].rename(
    columns={
        "net_export_mw": "baseline_net_export_mw",
        "afrr_up_reserved_mw": "baseline_afrr_up_reserved_mw",
        "afrr_down_reserved_mw": "baseline_afrr_down_reserved_mw",
        "lock_state": "baseline_lock_state",
        "schedule_version": "baseline_schedule_version",
    }
)
comparison = comparison.merge(
    revision[
        [
            "timestamp_local",
            "net_export_mw",
            "afrr_up_reserved_mw",
            "afrr_down_reserved_mw",
            "lock_state",
            "schedule_version",
        ]
    ].rename(
        columns={
            "net_export_mw": "revised_net_export_mw",
            "afrr_up_reserved_mw": "revised_afrr_up_reserved_mw",
            "afrr_down_reserved_mw": "revised_afrr_down_reserved_mw",
            "lock_state": "revised_lock_state",
            "schedule_version": "revised_schedule_version",
        }
    ),
    on="timestamp_local",
)
comparison["energy_changed"] = (comparison["baseline_net_export_mw"] - comparison["revised_net_export_mw"]).abs() > 1e-9
comparison["afrr_up_changed"] = (
    comparison["baseline_afrr_up_reserved_mw"] - comparison["revised_afrr_up_reserved_mw"]
).abs() > 1e-9
comparison["afrr_down_changed"] = (
    comparison["baseline_afrr_down_reserved_mw"] - comparison["revised_afrr_down_reserved_mw"]
).abs() > 1e-9

comparison.loc[
    comparison["energy_changed"] | comparison["afrr_up_changed"] | comparison["afrr_down_changed"],
    [
        "timestamp_local",
        "baseline_net_export_mw",
        "revised_net_export_mw",
        "baseline_afrr_up_reserved_mw",
        "revised_afrr_up_reserved_mw",
        "baseline_afrr_down_reserved_mw",
        "revised_afrr_down_reserved_mw",
        "baseline_lock_state",
        "revised_lock_state",
    ],
].head(12)

In [ ]:
pd.DataFrame(
    {
        "decision_events": [len(decision_log)],
        "lineage_rows": [len(lineage)],
        "schedule_versions": [sorted(lineage["schedule_version"].unique())],
        "revision_checkpoints": [
            sorted(
                {
                    str(value)
                    for value in decision_log.loc[
                        decision_log["decision_type"].astype(str).str.contains("revision", na=False),
                        "decision_time_local",
                    ]
                }
            )
        ],
    }
)

## Step 4 - Reconcile expected vs realized PnL

Reconciliation compares three layers: the baseline expected PnL, the revised expected PnL, and the realized PnL under the actual settlement inputs. For aFRR runs, the summary also includes an activation settlement deviation bucket.


In [ ]:
reconciliation_dir = reconcile_run(result.output_dir, CONFIG_PATH)
reconciliation_summary = json.loads((reconciliation_dir / "reconciliation_summary.json").read_text())
reconciliation_breakdown = pd.read_parquet(reconciliation_dir / "reconciliation_breakdown.parquet")

pd.DataFrame(
    [
        {
            "baseline_expected_total_pnl_eur": reconciliation_summary["baseline_expected_total_pnl_eur"],
            "revised_expected_total_pnl_eur": reconciliation_summary["revised_expected_total_pnl_eur"],
            "realized_total_pnl_eur": reconciliation_summary["realized_total_pnl_eur"],
            "delta_vs_baseline_expected_eur": reconciliation_summary["delta_vs_baseline_expected_eur"],
            "delta_vs_revised_expected_eur": reconciliation_summary["delta_vs_revised_expected_eur"],
            "forecast_error_eur": reconciliation_summary["forecast_error_eur"],
            "locked_commitment_opportunity_cost_eur": reconciliation_summary["locked_commitment_opportunity_cost_eur"],
            "activation_settlement_deviation_eur": reconciliation_summary["activation_settlement_deviation_eur"],
        }
    ]
)

In [ ]:
reconciliation_breakdown[
    [
        "timestamp_utc",
        "baseline_expected_pnl_eur",
        "revised_expected_pnl_eur",
        "realized_pnl_eur",
        "forecast_error_eur",
        "activation_settlement_deviation_eur",
    ]
].head(8)

## Step 5 - Export schedule, revision, and bid artifacts

The export layer keeps this benchmark useful outside the notebook. Even though these files are **not** live-submission payloads, they are stable handoff surfaces for traders, analysts, and downstream tooling.


In [ ]:
schedule_export = export_schedule(result.output_dir)
revision_export = export_revision(result.output_dir)
bid_export = export_bids(result.output_dir)

pd.DataFrame(
    [
        {
            "export": "schedule",
            "path": str(schedule_export),
            "files": sorted(path.name for path in schedule_export.glob("*.json")),
        },
        {
            "export": "revision",
            "path": str(revision_export),
            "files": sorted(path.name for path in revision_export.glob("*.json")),
        },
        {"export": "bids", "path": str(bid_export), "files": sorted(path.name for path in bid_export.glob("*.json"))},
    ]
)

## Wrap-up

This notebook demonstrates the current strongest `euroflex_bess_lab` path:

- a Belgium portfolio benchmark with asymmetric expected-value aFRR
- checkpoint-based schedule revision that preserves locked reserve commitments
- reconciliation against realized settlement inputs
- stable downstream exports for schedule and bid handoff
